# Random Arrays with NumPy

This notebook demonstrates reproducible random array generation, common distributions, shuffling, multivariate sampling, visualization, testing, and performance tips.

## 1. Import Required Libraries

Import NumPy, matplotlib, and helper libraries. Set up inline plotting for notebook use.

In [ ]:
import timeit

import matplotlib.pyplot as plt
import numpy as np

try:
    import seaborn as sns
except ImportError:
    sns = None

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

## 2. Create Random Arrays

Demonstrate 1D, 2D, and 3D random arrays and inspect shapes and dtypes.

In [ ]:
one_d = np.random.rand(5)
two_d = np.random.rand(3, 4)
three_d = np.random.rand(2, 3, 4)

print("1D:", one_d, one_d.shape, one_d.dtype)
print("2D:\n", two_d, two_d.shape, two_d.dtype)
print("3D shape:", three_d.shape, three_d.dtype)

## 3. Seed the Random Number Generator

Compare the legacy `np.random.seed` API with the newer `default_rng` generator for reproducibility.

In [ ]:
np.random.seed(42)
legacy_sample = np.random.rand(5)
np.random.seed(42)
legacy_repeat = np.random.rand(5)

rng = np.random.default_rng(42)
generator_sample = rng.random(5)
rng_repeat = np.random.default_rng(42).random(5)

print("Legacy reproducible:", np.allclose(legacy_sample, legacy_repeat))
print("Generator reproducible:", np.allclose(generator_sample, rng_repeat))
print("Legacy sample:", legacy_sample)
print("Generator sample:", generator_sample)

## 4. Generate Random Integers

Use `randint` and `Generator.integers` to generate integer arrays with clear bounds.

In [ ]:
legacy_ints = np.random.randint(1, 10, size=(3, 4))
generator_ints = np.random.default_rng(7).integers(1, 10, size=(3, 4))

print("Legacy integers:\n", legacy_ints)
print("Generator integers:\n", generator_ints)
print("Bounds are inclusive of low and exclusive of high:", legacy_ints.min() >= 1 and legacy_ints.max() < 10)

## 5. Generate Random Floats

Use `rand`, `random`, and `Generator.random` to produce floats in `[0, 1)` and scale them to custom ranges.

In [ ]:
legacy_floats = np.random.rand(2, 3)
legacy_scaled = 10 + 5 * np.random.rand(4)
generator_floats = np.random.default_rng(9).random((2, 3))

print("Legacy floats:\n", legacy_floats)
print("Scaled floats:", legacy_scaled)
print("Generator floats:\n", generator_floats)

## 6. Random Samples from Common Distributions

Draw samples from normal, binomial, Poisson, and exponential distributions and inspect summary statistics.

In [ ]:
normal = np.random.default_rng(123).normal(loc=0, scale=1, size=1000)
binomial = np.random.default_rng(123).binomial(n=10, p=0.4, size=1000)
poisson = np.random.default_rng(123).poisson(lam=3, size=1000)
exponential = np.random.default_rng(123).exponential(scale=2, size=1000)

print({
    "normal_mean": normal.mean(),
    "binomial_mean": binomial.mean(),
    "poisson_mean": poisson.mean(),
    "exponential_mean": exponential.mean(),
})

## 7. Shuffle and Permutations

Demonstrate in-place shuffling versus returned permutations.

In [ ]:
items = np.array(["A", "B", "C", "D", "E"])
permuted_items = np.random.default_rng(99).permutation(items)
shuffled_items = items.copy()
np.random.shuffle(shuffled_items)

print("Original:", items)
print("Permutation:", permuted_items)
print("Shuffled in place:", shuffled_items)

## 8. Multivariate Random Sampling

Use correlated sampling with a covariance matrix and draw from a Dirichlet distribution.

In [ ]:
mean = [0, 0]
cov = [[1, 0.8], [0.8, 1]]
multivariate = np.random.default_rng(1234).multivariate_normal(mean, cov, size=500)
dirichlet = np.random.default_rng(1234).dirichlet([2, 3, 5], size=5)

print("Multivariate shape:", multivariate.shape)
print("Dirichlet sample:\n", dirichlet)

## 9. Visualize Random Data

Plot histograms, density estimates, scatter plots, and pair-style views for sampled data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(normal, bins=30, density=True, alpha=0.7)
axes[0].set_title("Normal Histogram")
axes[1].scatter(multivariate[:, 0], multivariate[:, 1], s=10, alpha=0.5)
axes[1].set_title("Correlated Scatter")
axes[2].bar(range(dirichlet.shape[1]), dirichlet.mean(axis=0))
axes[2].set_title("Dirichlet Mean Components")
plt.tight_layout()
plt.show()

## 10. Reproducible Experiments and Unit Tests

Show small pytest-style assertions for shapes, ranges, and approximate statistical properties.

In [ ]:
def test_random_shape_and_bounds():
    sample = np.random.default_rng(7).integers(0, 5, size=(2, 3))
    assert sample.shape == (2, 3)
    assert sample.min() >= 0
    assert sample.max() < 5


def test_normal_mean_is_close_to_zero():
    sample = np.random.default_rng(7).normal(loc=0, scale=1, size=1000)
    assert abs(sample.mean()) < 0.2

print("Tests are defined and can be executed with pytest in a real test file.")

## 11. Performance and Memory Considerations

Compare generation strategies, consider `dtype` choices, and discuss chunked generation for large arrays.

In [ ]:
large_generation = timeit.timeit(lambda: np.random.default_rng(42).random((1000, 1000)), number=10)
loop_generation = timeit.timeit(lambda: [np.random.default_rng(42).random() for _ in range(1000)], number=10)

small_dtype = np.random.default_rng(42).random(1000).astype(np.float32)
print("Vectorized generation time:", large_generation)
print("Loop generation time:", loop_generation)
print("Float32 bytes:", small_dtype.nbytes)